# IMAEP de Paraguay: análisis exploratorio 2016-2025

[Abrir este notebook en Google Colab](https://colab.research.google.com/github/brunoaguilera/kit_codex_imaep/blob/main/docs/imaep_colab.ipynb)

Notebook reproducible para Google Colab. Carga la serie oficial procesada, muestra las 120 observaciones, recalcula las estadísticas y presenta las imágenes de la primera entrega. No realiza modelos ni pronósticos.

In [ ]:
from pathlib import Path
import json
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
from IPython.display import Image, display

pd.set_option('display.max_rows', None)
pd.set_option('display.precision', 12)


In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/brunoaguilera/kit_codex_imaep/main'
csv_url = f'{BASE_URL}/data/processed/imaep_2016_2025.csv'
quality_url = f'{BASE_URL}/outputs/calidad_datos.json'
df = pd.read_csv(csv_url, parse_dates=['fecha'])
quality = requests.get(quality_url, timeout=30).json()
assert quality['estado'] == 'OK'
assert len(df) == 120
print(f'Observaciones: {len(df)}')
print(f"Período: {df.fecha.min().date()} a {df.fecha.max().date()}")


## Datos completos

In [ ]:
display(df)


## Controles de calidad registrados

In [ ]:
controles = pd.Series(quality['controles'], name='resultado')
display(controles.to_frame())


## Estadísticas básicas

In [ ]:
serie = df['imaep']
q1 = serie.quantile(0.25, interpolation='linear')
q3 = serie.quantile(0.75, interpolation='linear')
varianza = serie.var(ddof=1)
desviacion = serie.std(ddof=1)
estadisticas = pd.Series({
    'cantidad_valida': int(serie.notna().sum()),
    'cantidad_faltante': int(serie.isna().sum()),
    'media': serie.mean(),
    'mediana': serie.median(),
    'minimo': serie.min(),
    'mes_minimo': df.loc[serie.eq(serie.min()), 'fecha'].dt.strftime('%Y-%m-%d').tolist(),
    'maximo': serie.max(),
    'mes_maximo': df.loc[serie.eq(serie.max()), 'fecha'].dt.strftime('%Y-%m-%d').tolist(),
    'rango': serie.max() - serie.min(),
    'q1': q1,
    'q3': q3,
    'rango_intercuartilico': q3 - q1,
    'varianza_muestral_n_1': varianza,
    'desviacion_estandar_muestral_n_1': desviacion,
})
assert math.isclose(desviacion ** 2, varianza, rel_tol=1e-12, abs_tol=1e-12)
display(estadisticas.to_frame('valor'))


## Gráfica recalculada en Colab

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df['fecha'], df['imaep'], color='#1f4e79', linewidth=1.8)
ax.set_title('Paraguay: IMAEP total, serie original mensual (2016-2025)')
ax.set_xlabel('Mes')
ax.set_ylabel('Índice (base 2014 = 100)')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.grid(axis='y', color='#d9d9d9')
plt.show()


## Imagen entregada

In [ ]:
display(Image(url=f'{BASE_URL}/outputs/serie_imaep.png', width=1000))


## Flujo metodológico de referencia

La imagen incluye etapas posteriores de series temporales. En esta entrega se implementaron únicamente datos, controles y exploración descriptiva.

In [ ]:
display(Image(url=f'{BASE_URL}/referencias/flujo_series_temporales.jpg', width=1000))


## Alcance

Las cifras están marcadas como preliminares y sujetas a revisión. La descripción no prueba estacionalidad o estacionariedad y no atribuye causas a las variaciones. No se realizan imputaciones, modelos ni pronósticos.